In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install -q fastapi uvicorn pyngrok accelerate faiss-cpu langchain langchain-community langchain-core langchain-huggingface langchain-text-splitters sentence-transformers transformers==4.52.4


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 82.1 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 91.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 67.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.8/572.8 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 85.8 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "mistralai/Mistral-Nemo-Instruct-2407"  


tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
).eval()


def generate_text(prompt, max_length=600):
    """Send prompt to Mistral and return ONLY the newly generated text."""
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_length,
            do_sample=False,                 
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

model-00003-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

model-00005-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [4]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import CharacterTextSplitter

embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
splitter = CharacterTextSplitter(separator="\n", chunk_size=1000, chunk_overlap=100)


def build_index(transcript):
    chunks = splitter.split_text(transcript)
    if not chunks:
        chunks = [transcript]
    return chunks, FAISS.from_texts(chunks, embedding)


def retrieve(vectordb, query, k=4):
    docs = vectordb.similarity_search(query, k=k)
    return "\n\n".join(d.page_content for d in docs)


/tmp/ipykernel_49/2589717282.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
import json, re
from langchain_core.prompts import PromptTemplate

try:
    from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema
except ImportError:
    from langchain.output_parsers import StructuredOutputParser, ResponseSchema


summary_parser = StructuredOutputParser.from_response_schemas([
    ResponseSchema(name="summary", description="A short summary of the meeting in 3-5 sentences."),
])


decisions_parser = StructuredOutputParser.from_response_schemas([
    ResponseSchema(name="decisions",
                   description="A list of strings. Each string is one decision the team agreed on in the meeting."),
])


tasks_parser = StructuredOutputParser.from_response_schemas([
    ResponseSchema(
        name="tasks",
        description=("A list of objects. Each object has exactly these keys: "
                     "'task' (what must be done), "
                     "'owner' (person responsible, or 'Unassigned'), "
                     "'deadline' (due date/time mentioned, or 'Not specified')."),
    ),
])


def extract_json_block(text):
    """Get the JSON from the model answer and wrap it in ```json ``` so the parser accepts it."""
    m = re.findall(r"```json\s*(.*?)\s*```", text, re.DOTALL)
    if m:
        body = m[-1]
    else:
        start, end = text.find("{"), text.rfind("}")
        body = text[start:end + 1] if start != -1 and end != -1 else "{}"
    return f"```json\n{body}\n```"


def parse_output(parser, raw_text, key, default):
    block = extract_json_block(raw_text)
    try:
        return parser.parse(block)[key]
    except Exception:
        try:  
            return json.loads(block.strip("`").replace("json\n", "", 1))[key]
        except Exception:
            return default

In [6]:
summary_template = """You are an assistant that analyzes meeting transcripts.
Write a clear short summary of the meeting below.

Respond ONLY in JSON format as follows:
{format_instructions}

Meeting transcript:
{context}
"""

decisions_template = """You are an assistant that analyzes meeting transcripts.
Extract ONLY the decisions that were clearly agreed on. Do not invent anything.
If there are no decisions, return an empty list.

Respond ONLY in JSON format as follows:
{format_instructions}

Relevant parts of the meeting transcript:
{context}
"""

tasks_template = """You are an assistant that analyzes meeting transcripts.
Extract ALL action items / tasks. For every task give the owner (who will do it)
and the deadline (when it is due). Use 'Unassigned' or 'Not specified' if missing.
Do not invent anything.

Respond ONLY in JSON format as follows:
{format_instructions}

Relevant parts of the meeting transcript:
{context}
"""


def run_chain(template, parser, context, max_length=600):
    prompt = PromptTemplate(
        template=template, input_variables=["context", "format_instructions"]
    ).format(context=context, format_instructions=parser.get_format_instructions())
    return generate_text(prompt, max_length=max_length)


def summarize_meeting(transcript, chunks, vectordb):
    
    if len(transcript) <= 6000:
        context = transcript
    else:
        partial = []
        for c in chunks:
            partial.append(generate_text(f"Summarize this part of a meeting in 2 sentences:\n\n{c}", 150))
        context = "\n".join(partial)
    raw = run_chain(summary_template, summary_parser, context, 400)
    return parse_output(summary_parser, raw, "summary", raw)


def extract_decisions(vectordb):
    ctx = retrieve(vectordb, "decisions agreed approved decided we will go with final choice", k=4)
    raw = run_chain(decisions_template, decisions_parser, ctx, 500)
    return parse_output(decisions_parser, raw, "decisions", [])


def extract_tasks(vectordb):
    ctx = retrieve(vectordb, "action items tasks assigned to responsible deadline due by will do", k=5)
    raw = run_chain(tasks_template, tasks_parser, ctx, 700)
    return parse_output(tasks_parser, raw, "tasks", [])


def analyze_meeting(transcript):
    chunks, vectordb = build_index(transcript)
    return {
        "summary": summarize_meeting(transcript, chunks, vectordb),
        "decisions": extract_decisions(vectordb),
        "tasks": extract_tasks(vectordb),
    }


In [7]:
NGROK_TOKEN = "3KNvrzhB4HBVGm3enkvCX4TEo4R_7FKbX9FXQK6koYZJMnLRB"
API_KEY = "secret123"

from fastapi import FastAPI, Request, HTTPException
import uvicorn, threading, time, socket, uuid, traceback
from pyngrok import ngrok, conf

ngrok.kill()  

app = FastAPI()
jobs = {}                      
gpu_lock = threading.Lock()    


def check_auth(req):
    if req.headers.get("authorization") != f"Bearer {API_KEY}":
        raise HTTPException(status_code=401, detail="Unauthorized")


def worker(job_id, transcript):
    try:
        with gpu_lock:
            jobs[job_id]["status"] = "running"
            t = time.time()
            result = analyze_meeting(transcript)
            print(f"Job {job_id[:6]} finished in {time.time() - t:.0f}s")
        jobs[job_id].update(status="done", result=result)
    except Exception:
        err = traceback.format_exc()
        print(err)  
        jobs[job_id].update(status="error", error=err)


@app.get("/health")
async def health():
    return {"ok": True}


@app.post("/generate")
async def gen(req: Request):
    check_auth(req)
    data = await req.json()
    return {"response": generate_text(data.get("prompt", ""), data.get("max_length", 300))}


@app.post("/analyze")
async def analyze(req: Request):
    check_auth(req)
    data = await req.json()
    transcript = data.get("transcript", "").strip()
    if not transcript:
        raise HTTPException(status_code=400, detail="Empty transcript")
    job_id = uuid.uuid4().hex
    jobs[job_id] = {"status": "queued"}
    threading.Thread(target=worker, args=(job_id, transcript), daemon=True).start()
    return {"job_id": job_id}


@app.get("/result/{job_id}")
async def result(job_id: str, req: Request):
    check_auth(req)
    if job_id not in jobs:
        raise HTTPException(status_code=404, detail="Unknown job")
    return jobs[job_id]


def free_port():
    s = socket.socket()
    s.bind(("", 0))
    port = s.getsockname()[1]
    s.close()
    return port


port = free_port()
conf.get_default().auth_token = NGROK_TOKEN
public_url = ngrok.connect(port).public_url
print("Your public URL:", public_url)


def run():
    uvicorn.run(app, host="0.0.0.0", port=port)


threading.Thread(target=run, daemon=True).start()
time.sleep(2)


import requests
print(requests.get(f"http://127.0.0.1:{port}/health").json())

Your public URL: https://vineyard-evade-tarantula.ngrok-free.dev                                    


INFO:     Started server process [49]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:47871 (Press CTRL+C to quit)


INFO:     127.0.0.1:43406 - "GET /health HTTP/1.1" 200 OK
{'ok': True}
INFO:     149.154.161.219:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     156.204.116.207:0 - "POST /analyze HTTP/1.1" 200 OK
INFO:     156.204.116.207:0 - "GET /result/c82a489ebfba45b7b8a989d6c4f0da38 HTTP/1.1" 200 OK
INFO:     156.204.116.207:0 - "GET /result/c82a489ebfba45b7b8a989d6c4f0da38 HTTP/1.1" 200 OK
INFO:     156.204.116.207:0 - "GET /result/c82a489ebfba45b7b8a989d6c4f0da38 HTTP/1.1" 200 OK
INFO:     156.204.116.207:0 - "GET /result/c82a489ebfba45b7b8a989d6c4f0da38 HTTP/1.1" 200 OK
INFO:     156.204.116.207:0 - "GET /result/c82a489ebfba45b7b8a989d6c4f0da38 HTTP/1.1" 200 OK
INFO:     156.204.116.207:0 - "GET /result/c82a489ebfba45b7b8a989d6c4f0da38 HTTP/1.1" 200 OK
INFO:     156.204.116.207:0 - "GET /result/c82a489ebfba45b7b8a989d6c4f0da38 HTTP/1.1" 200 OK
INFO:     156.204.116.207:0 - "GET /result/c82a489ebfba45b7b8a989d6c4f0da38 HTTP/1.1" 200 OK
INFO:     156.204.116.207:0 - "GET /result/c82a489ebfba45b7b8a